In [13]:
import os, uuid, hashlib, platform, getpass
# ===== 1. 填写学号 =====
STUDENT_ID = 20244081101 #修改为自己的学号
# ===== 2. 打印 conda 环境 =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")
# ===== 3. 机器指纹（绑定当前电脑）=====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]\t{STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")


[Conda 环境] pyexp
[学号]	20244081101
[机器指纹] 3cf3fba9863bfd88
[RUN_ID] 20244081101-3cf3fba9863bfd88


## 热身：定义简单函数

这三个函数都负责打印，不返回值。`goat_pad` 在原字符串两侧各接上一次 `goat`，中间不加空格。`print_n` 把同一个字符串打印 `n` 行。


In [14]:
def say_hi():
    print("Hello, world!")


def goat_pad(text):
    print("goat" + text + "goat")


def print_n(s, n):
    for _ in range(n):
        print(s)


say_hi()
goat_pad("bird")
goat_pad("_")
print_n("ha", 3)


Hello, world!
goatbirdgoat
goat_goat
ha
ha
ha


## 欧几里得算法

两个整数的最大公约数，是能同时整除它们的最大正整数。这里用辗转相除：用 `m % n` 替换较大的数，直到余数为 0，此时另一个数就是公约数。题目允许输入都是整数，所以函数内部不做类型检查。


In [15]:
def gcd(m, n):
    while n != 0:
        m, n = n, m % n
    return m


pairs = [(2018, 2019), (1200, 300), (5040, 60)]
for m, n in pairs:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")


gcd(2018, 2019) = 1
gcd(1200, 300) = 300
gcd(5040, 60) = 60


如果参数是负数，循环仍然会结束，因为 Python 的 `%` 结果与除数同号，余数的绝对值会变小。返回值却可能是负数。例如两个负数的公约数会被算成负的。

这和“能整除”并不矛盾：如果 `d` 能整除两个数，`-d` 也能整除它们。但数学上的最大公约数约定为非负整数，所以直接返回负数并不符合通常定义。这个结果来自 Python 取余的符号规则，而不是算法本身希望得到负数。若需要非负结果，可以在返回前取绝对值。下面的实现保持算法原样，用来观察这种现象。


In [16]:
for m, n in ((20, 10), (-20, 10), (20, -10), (-20, -10), (-8, -12)):
    print(f"gcd({m}, {n}) = {gcd(m, n)}")


gcd(20, 10) = 10
gcd(-20, 10) = 10
gcd(20, -10) = -10
gcd(-20, -10) = -10
gcd(-8, -12) = -4


## 近似欧拉数 e

e 可以写成无穷级数

\(e = \sum_{k=0}^{\infty} 1/k! = 1 + 1 + 1/2 + 1/6 + 1/24 + \cdots\)

其中 \(0! = 1\)。伯努利给出的另一种定义是极限

\(e = \lim_{x \to \infty} (1 + 1/x)^x\)。

`euler_infinite_sum(n)` 取级数的前 `n` 项。级数本身从 `k = 0` 开始数，函数参数从 1 开始数：`n = 1` 只用第一项，结果是 1；`n = 0` 是空和，结果是 0。因此 `euler_infinite_sum(4)` 等于 `1 + 1 + 1/2 + 1/6`。


In [17]:
import math


def euler_limit(n):
    return float((1 + 1 / n) ** n)


def euler_infinite_sum(n):
    total = 0.0
    term = 1.0
    for k in range(n):
        if k > 0:
            term /= k
        total += term
    return total


def euler_approx(epsilon):
    true_e = math.exp(1)
    total = 0.0
    term = 1.0
    k = 0
    while abs(true_e - total) >= epsilon:
        next_total = total + term
        if next_total == total:
            break
        total = next_total
        k += 1
        term /= k
    return float(total)


def print_euler_sum_table(n):
    for k in range(1, n + 1):
        print(euler_infinite_sum(k))


print("euler_infinite_sum(0) =", euler_infinite_sum(0))
print("euler_infinite_sum(1) =", euler_infinite_sum(1))
four_terms = euler_infinite_sum(4)
expected = 1 + 1 + 1 / 2 + 1 / 6
print("euler_infinite_sum(4) =", four_terms)
print("与 1+1+1/2+1/6 一致:", abs(four_terms - expected) < 1e-12)

print("极限近似:")
for n in (1, 10, 100, 1000, 100000):
    approx = euler_limit(n)
    print(f"n={n:<6}  {approx:.10f}  与 math.e 的差={abs(approx - math.e):.3e}")

for epsilon in (0.1, 1e-6, 1e-10):
    approx = euler_approx(epsilon)
    error = abs(approx - math.exp(1))
    print(f"epsilon={epsilon:<8}  近似={approx:.12f}  误差={error:.3e}  达标={error < epsilon}")

print("求和表 n=6:")
print_euler_sum_table(6)


euler_infinite_sum(0) = 0.0
euler_infinite_sum(1) = 1.0
euler_infinite_sum(4) = 2.6666666666666665
与 1+1+1/2+1/6 一致: True
极限近似:
n=1       2.0000000000  与 math.e 的差=7.183e-01
n=10      2.5937424601  与 math.e 的差=1.245e-01
n=100     2.7048138294  与 math.e 的差=1.347e-02
n=1000    2.7169239322  与 math.e 的差=1.358e-03
n=100000  2.7182682372  与 math.e 的差=1.359e-05
epsilon=0.1       近似=2.666666666667  误差=5.162e-02  达标=True
epsilon=1e-06     近似=2.718281525573  误差=3.029e-07  达标=True
epsilon=1e-10     近似=2.718281828447  误差=1.229e-11  达标=True
求和表 n=6:
1.0
2.0
2.5
2.6666666666666665
2.708333333333333
2.7166666666666663


`euler_approx` 用 while 循环不断补上下一项，直到近似值和 `math.exp(1)` 的差距小于给定的 `epsilon`。浮点数精度有限，某一项小到已经不能再改变当前和时就停止，这时得到的是这个级数在浮点运算下最接近 e 的结果。

`print_euler_sum_table(n)` 把 `k = 1, 2, ..., n` 的部分和逐行打印出来，可以看到项数增加后结果怎样靠近 e。


## 测试整数属性

正整数 `n` 是 2 的幂，是指存在整数 `p ≥ 0`，使得 `n = 2^p`。因此 1 也算，因为 `1 = 2^0`。判断时只用除法和取余：能被 2 整除就不断除以 2，最后恰好得到 1 才是 2 的幂。

`is_power(b, n)` 把这个过程推广到任意底数：不断用 `b` 去除 `n`，看能不能正好除到 1。`b == 1` 要单独处理，因为任何整数除以 1 都余 0，但除完数值不变，循环不会结束；而 1 的正整数次幂只有 1 本身。`n == 1` 时直接返回 True，因为任何正整数的 0 次幂都是 1。


In [18]:
def is_power_of_2(n):
    if n == 1:
        return True
    while n % 2 == 0:
        n //= 2
        if n == 1:
            return True
    return False


def is_power(b, n):
    if n == 1:
        return True
    if b == 1:
        return False
    while n % b == 0:
        n //= b
        if n == 1:
            return True
    return False


print("2 的幂:")
for n in (1, 2, 3, 4, 6, 8, 12, 16, 18, 32):
    print(f"is_power_of_2({n}) = {is_power_of_2(n)}")

print("与 is_power(2, n) 一致:", all(
    is_power_of_2(n) == is_power(2, n) for n in range(1, 65)
))

cases = [(2, 8), (2, 7), (3, 27), (3, 28), (5, 1), (1, 1), (1, 8), (10, 1000), (10, 100), (4, 2), (9, 81)]
for b, n in cases:
    print(f"is_power({b}, {n}) = {is_power(b, n)}")


2 的幂:
is_power_of_2(1) = True
is_power_of_2(2) = True
is_power_of_2(3) = False
is_power_of_2(4) = True
is_power_of_2(6) = False
is_power_of_2(8) = True
is_power_of_2(12) = False
is_power_of_2(16) = True
is_power_of_2(18) = False
is_power_of_2(32) = True
与 is_power(2, n) 一致: True
is_power(2, 8) = True
is_power(2, 7) = False
is_power(3, 27) = True
is_power(3, 28) = False
is_power(5, 1) = True
is_power(1, 1) = True
is_power(1, 8) = False
is_power(10, 1000) = True
is_power(10, 100) = True
is_power(4, 2) = False
is_power(9, 81) = True
